# Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT)

Prepared by:
: J M Maxwell, Center for Translational Data Science (CTDS), University of Chicago

:::{note} Attribution
:class: dropdown
This notebook uses data collected by [](doi:10.1001/jamanetworkopen.2024.9744) as part of their study entitled *Surmounting Withdrawal to Initiate Fast Treatment with Naltrexone (SWIFT): Improving the Real-World Effectiveness of Injection Naltrexone for Opioid Use Disorder*. The data have been archived by the authors at [NIDA Data Share](https://datashare.nida.nih.gov/study/nida-ctn-0097) and are accessible via the [HEAL Data Platform](10.60490/HDP01319).

The purpose of this notebook is to demonstrate how the data may be accessed and used for analysis, and is intended to be used as a jumping off point for researchers who may wish to use these data for their own secondary analyses. While some of the analyses below may recreate analyses presented in the original publication, they are not intended to replicate the original results and differences in analytic methods and software, selection/filtering of observations, and handling of missing data may yield results that differ from the original.

The work here was conducted without direct involvement of the original authors and therefore does not necessarily reflect the views or opinions of the authors, of the NIH HEAL Initiative&reg;, or of the Center for Translational Data Science (CTDS) at the University of Chicago.
:::

## About The Study

The study, *Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT)*, investigates two methods of treatment with extended-release naltrexone (XR-NTX). The primary comparison between the two methods is if the Rapid Method treatment (5-7 days long) is non-inferior to the Standard Method (13 days long) in succesful initiation of XR-NTX, with a secondary comparisons on time from admission to dose to dropout, craving, withdrawal severity, retention, abstinence, and safety measures as measured from admission through the first two months of post-induction maintenance of XR-NTX. [(NCT: 04762537)](https://clinicaltrials.gov/study/NCT04762537) 

The Standard Procedure was defined as an approximately 5 day buprenorphine taper, a 7-10 day opiod free period, finished with a XR-naltrexone package insert. The Rapid Proceudre was defined as 1 day of buprenorphine at the minimal necessary dosage, 1 day opioid free, finished with ascending low dosages of oral naltrexone and additional medications for opioid withdrawal.

## Setup

This notebook uses Python (tested with version 3.13) and relies on several modules and third-party packages. Thus we start by installing and/or importing these if necessary (click on the bar below to expand the cell and see the code).

In [ ]:
!pip install pandas openpyxl gen3 tableone -q
import os
import pandas as pd

## Accessing the Data

To access the data from this study make sure you are logged in to the InCommon login option and then: 
1) Go to the HEAL Discovery page to select the study
2) Select the 'Open In Workspace' option and choose the (Tutorials) Example Analysis Jupyter Lab Notebooks workspace option
3) Use the exported study manifest to download the study. 

Otherwise you may run the following gen3-sdk command to download the relevant files from the study.

In [ ]:
if not os.path.exists('sas-crf-data-files_nida-ctn-0097 v1-1.zip'):
    !gen3 drs-pull object dg.H34L/8a0b2f16-4bd1-4033-acfe-b353917aeb20

## Import Packages And Read Data

Import necessary python packages and read in the data for the core variables and inpatient COWS scores.

In [ ]:
!pip install openpyxl -q
!pip install matplotlib -q


import os
import shutil
import zipfile

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt

from IPython.display import Markdown, Image, display

folder_path = 'img/SWIFT_OUD_Treatment'
if os.path.exists(folder_path) and os.path.isdir(folder_path):
    shutil.rmtree(folder_path)
    os.makedirs(folder_path)
    print(f"A new folder - '{folder_path}' - has been created.")
else:
    os.makedirs(folder_path)
    print(f"A new folder - '{folder_path}' - has been created.")

In [ ]:
with zipfile.ZipFile("sas-crf-data-files_nida-ctn-0097 v1-1.zip", 'r') as zip_ref:
        zip_ref.extractall("ctn0097")
    
df1 = pd.read_sas("ctn0097/corevars.sas7bdat")
df2 = pd.read_sas("ctn0097/inpatient.sas7bdat", encoding='utf-8')

In [ ]:
def cat_value_counts(df, field, map_dict):

    results = {'Standard (n=190)': {}, 'Rapid (n=225)': {}, 'Total (N=415)': {}}

    l1 = df[df.TRT == b'Standard'][field].value_counts().to_dict()
    s1 = sum(l1.values())
    for k in map_dict.keys():
        results['Standard (n=190)'][map_dict[k]] = f"{l1[k]} ({np.round(100*l1[k]/s1, 1)})"
        
    l2 = df[df.TRT == b'Rapid'][field].value_counts().to_dict()
    s2 = sum(l2.values())
    for k in map_dict.keys():
        results['Rapid (n=225)'][map_dict[k]] = f"{l2[k]} ({np.round(100*l2[k]/s2, 1)})"
        
    l3 = df[field].value_counts().to_dict()
    s3 = sum(l3.values())
    for k in map_dict.keys():
        results['Total (N=415)'][map_dict[k]] = f"{l3[k]} ({np.round(100*l3[k]/s3, 1)})"
    
    return results
    

def num_values(df, field):

    results = {'Standard (n=190)': {}, 'Rapid (n=225)': {}, 'Total (N=415)': {}}

    avg = df[df.TRT == b'Standard'][field].mean().round(1)
    std = round(df[df.TRT == b'Standard'][field].std(), 1)
    results['Standard (n=190)']['Mean (SD)'] = f"{avg} ({std})"

    avg = df[df.TRT == b'Rapid'][field].mean().round(1)
    std = round(df[df.TRT == b'Rapid'][field].std(), 1)
    results['Rapid (n=225)']['Mean (SD)'] = f"{avg} ({std})"

    avg = df[field].mean().round(1)
    std = round(df[field].std(), 1)
    results['Total (N=415)']['Mean (SD)'] = f"{avg} ({std})"
    
    return results

## Secondary Outcome Analysis

While the primary outcome measured was the receipt of the first dose of XR-naltrexone during the inpatient period, the secondary outcome explore here is the Clinical Opiate Withdrawal Scalse (COWS) Scores over the first 15 days of the trial. 

In the table below we see that 415 individuals participated in the study and the corresponding demographic summaries of the individuals.

141 of the 225 (62.7%) assigned to the Rapid Procedure and 68 of the 190 (35.8%) assigned to the Standard Procedure ultimately received XR-naltrexone. Through non-inferiority tests and subgroup analyses it was determined that the rate of initiation success among the Rapid Procedure group was noninferior to the rate of initiation in the Standard Procedure group.


In [ ]:
results1 = num_values(df1, 'AGE')

results2 = cat_value_counts(df1, 
                           'SEX', 
                           {1.0: 'Male', 2.0: 'Female'}
                          )
results3 = cat_value_counts(df1, 
                           'ETHNIC_CAT', 
                           {1.0: 'Hispanic'}
                          )
results4 = cat_value_counts(df1, 
                           'RACE', 
                           {1.0: 'Indigenous American', 2.0: 'Asian', 3.0: 'Black or African American', 
                            4.0: 'Hawaiian or Pacific Islander', 5.0: 'White', 7.0: 'Multiracial'}
                          )

table = pd.concat([pd.DataFrame(results1), pd.DataFrame(results2), pd.DataFrame(results3), pd.DataFrame(results4)], keys=['Age', 'Sex', 'Ethnicity', 'Race'])

Markdown(table.to_markdown())

The COWS score was measured as a binary for the presence of at least 1 moderate or above COWS score. In the figure and table below we can see that the mean and standard deviations of the Rapid and Standard Procedure were very similar throughout the course of the inpatient days, and that withdrawal did not differ significantly between procedure conditions. 

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 6))

df2_viz = pd.DataFrame(df2.groupby(['INP_DAY', 'TRT'])['COW_SCORE_MAX'].agg(['mean', 'std'])).reset_index()
df2_viz = df2_viz[(df2_viz.INP_DAY > 0) & (df2_viz.INP_DAY <= 15)]

plt.errorbar(df2_viz[df2_viz.TRT == 'Standard'].INP_DAY - 0.1, df2_viz[df2_viz.TRT == 'Standard']['mean'],  df2_viz[df2_viz.TRT == 'Standard']['std'], marker='.', elinewidth=0.9)
plt.errorbar(df2_viz[df2_viz.TRT == 'Rapid'].INP_DAY + 0.1, df2_viz[df2_viz.TRT == 'Rapid']['mean'], df2_viz[df2_viz.TRT == 'Rapid']['std'], marker='.', elinewidth=0.9)


plt.xticks(np.linspace(1, 15, 15))
plt.yticks(np.linspace(0, 15, 6))
plt.ylabel('Daily maximum COWS Score, Mean')
plt.xlabel('Inpatient Day')

plt.title('Mean (SD) Daily Maximum COWS Score During the First 15 Days Treatment Procedure.')

plt.legend(["Standard Procedure", "Rapid Procedure"])

plt.close(fig)

fig.savefig('img/SWIFT_OUD_Treatment/figure1.png')

Image(filename="img/SWIFT_OUD_Treatment/figure1.png")

In [ ]:
df2_viz = df2_viz.round(2)
Markdown(df2_viz.T.to_markdown())

## Conclusions

As we have briefly shown in this notebook, the study by Bisaga et. al. and the paper by Shulman et. al. demonstrates how the Rapid Procedure for XR-naltrexone initiation was noninferior compared to the Standard Procedure in treatment intiation and patients' self reported withdrawal outcomes. Shulman et. al. conclude that the Rapid Procedure treatment initiation, while requiring more intensive medical engagement and monitoring, "could make XR-naltrexone a more viable treatment for patients with OUD."

## References

#### Paper
Shulman M, Greiner MG, Tafessu HM, et al. Rapid Initiation of Injection Naltrexone for Opioid Use Disorder: A Stepped-Wedge Cluster Randomized Clinical Trial. JAMA Netw Open. 2024;7(5):e249744. Published 2024 May 1. doi:10.1001/jamanetworkopen.2024.9744

#### Study

Bisaga A, Nunes E. Surmounting Withdrawal to Initiate Fast Treatment With Naltrexone (SWIFT). NIDA-CTN-0097. https://datashare.nida.nih.gov/study/nida-ctn-0097

#### HEAL Data Platform

Bisaga, Adam & Jr.,   (2025). Surmounting Withdrawal to Initiate Fast Treatment with Naltrexone (SWIFT): Improving the Real-World Effectiveness of Injection Naltrexone for Opioid Use Disorder 2. HEAL Data Platform. Study Record. 10.60490/HDP01319